# C08 Compare challengers and all models

Build a single leaderboard for challenger runs and a wider overview that also includes the already evaluated legacy models from `city_swap_batch_eval`.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

CWD = Path.cwd()
NOTEBOOKS_DIR = CWD.parent if CWD.name == "challengers" else CWD
PROJECT_ROOT = NOTEBOOKS_DIR.parent

FIGURES_DIR = PROJECT_ROOT / "figures" / "challengers"
MODEL_DIR_CANDIDATES = [
    PROJECT_ROOT / "notebooks" / "models" / "challengers",
    PROJECT_ROOT / "models" / "challengers",
]
LEGACY_RESULTS_JSON = PROJECT_ROOT / "results" / "city_swap_all" / "city_swap_all_models.json"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
for model_dir in MODEL_DIR_CANDIDATES:
    model_dir.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"FIGURES_DIR: {FIGURES_DIR}")
print("MODEL_DIR_CANDIDATES:")
for model_dir in MODEL_DIR_CANDIDATES:
    print(f"  - {model_dir}")
print(f"LEGACY_RESULTS_JSON exists: {LEGACY_RESULTS_JSON.exists()}")


In [ ]:
SUMMARY_FILE_META = {
    "c04_class_balanced_tuning_summary.csv": {"family": "challenger", "notebook": "c04", "method_group": "class_balanced"},
    "c05_logit_adjustment_tuning_summary.csv": {"family": "challenger", "notebook": "c05", "method_group": "logit_adjustment"},
    "c06_weighted_sampler_tuning_summary.csv": {"family": "challenger", "notebook": "c06", "method_group": "weighted_sampler"},
    "c07_rdrop_tuning_summary.csv": {"family": "challenger", "notebook": "c07", "method_group": "rdrop"},
}

LEGACY_MODEL_REGISTRY = {
    "1_baseline": {"model_dir": "bert_9classes_final", "family": "legacy"},
    "2_groupdro": {"model_dir": "bert_gdro_eta01_2ep", "family": "legacy"},
    "3_scrubbing": {"model_dir": "bert_scrubbing", "family": "legacy"},
    "4_oversampling": {"model_dir": "bert_oversample_only", "family": "legacy"},
    "5_focal_loss": {"model_dir": "bert_focal_loss", "family": "legacy"},
    "6_adversarial": {"model_dir": "bert_adversarial", "family": "legacy"},
    "7_label_smoothing": {"model_dir": "bert_label_smoothing", "family": "legacy"},
    "8_attribution_reg": {"model_dir": "bert_attr_reg", "family": "legacy"},
    "9_combined_scrub_gdro": {"model_dir": "bert_debiased_combo", "family": "legacy"},
    "10_combined_best": {"model_dir": "combined_scrubbing_groupdro", "family": "legacy"},
}


def _safe_float(value):
    if value is None:
        return np.nan
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def _method_group_from_name(model_name):
    prefixes = [
        "class_balanced",
        "logit_adjustment",
        "weighted_sampler",
        "rdrop",
        "groupdro",
        "focal",
        "label_smoothing",
    ]
    for prefix in prefixes:
        if str(model_name).startswith(prefix):
            return prefix
    return "other"


def load_challenger_rows():
    rows = []
    summary_paths = sorted(FIGURES_DIR.glob("*_summary.csv"))

    for summary_path in summary_paths:
        try:
            df = pd.read_csv(summary_path)
        except Exception as exc:
            print(f"Skipping {summary_path.name}: {exc}")
            continue

        meta = SUMMARY_FILE_META.get(summary_path.name, {})
        for row in df.to_dict(orient="records"):
            model_name = row.get("model_name", summary_path.stem.replace("_summary", ""))
            model_dir = row.get("model_dir")
            config = {}
            if model_dir:
                config_path = PROJECT_ROOT / model_dir / "training_config.json"
                if config_path.exists():
                    config = json.loads(config_path.read_text())

            rows.append({
                "family": meta.get("family", "challenger"),
                "notebook": meta.get("notebook", summary_path.name.split("_")[0]),
                "method_group": meta.get("method_group", _method_group_from_name(model_name)),
                "model_name": model_name,
                "method": config.get("method"),
                "accuracy": _safe_float(row.get("accuracy", config.get("accuracy"))),
                "macro_f1": _safe_float(row.get("macro_f1", config.get("macro_f1"))),
                "worst_gap": _safe_float(row.get("worst_gap", config.get("tpr_gap_worst_robust"))),
                "macro_gap": _safe_float(row.get("macro_gap", config.get("tpr_gap_macro_robust"))),
                "overall_flip_rate": np.nan,
                "model_dir": model_dir,
                "summary_source": str(summary_path.relative_to(PROJECT_ROOT)),
            })

    if rows:
        return pd.DataFrame(rows)

    seen_model_names = set()
    for models_root in MODEL_DIR_CANDIDATES:
        for config_path in sorted(models_root.glob("*/training_config.json")):
            config = json.loads(config_path.read_text())
            model_name = config_path.parent.name
            if model_name in seen_model_names:
                continue
            seen_model_names.add(model_name)
            rows.append({
                "family": "challenger",
                "notebook": model_name.split("_")[0],
                "method_group": _method_group_from_name(model_name),
                "model_name": model_name,
                "method": config.get("method"),
                "accuracy": _safe_float(config.get("accuracy")),
                "macro_f1": _safe_float(config.get("macro_f1")),
                "worst_gap": _safe_float(config.get("tpr_gap_worst_robust")),
                "macro_gap": _safe_float(config.get("tpr_gap_macro_robust")),
                "overall_flip_rate": np.nan,
                "model_dir": str(config_path.parent.relative_to(PROJECT_ROOT)),
                "summary_source": str(config_path.relative_to(PROJECT_ROOT)),
            })

    return pd.DataFrame(rows)


def load_legacy_rows():
    if not LEGACY_RESULTS_JSON.exists():
        return pd.DataFrame()

    payload = json.loads(LEGACY_RESULTS_JSON.read_text())
    rows = []
    for model_name, metrics in payload.items():
        if isinstance(metrics, dict) and "error" in metrics:
            continue
        reg = LEGACY_MODEL_REGISTRY.get(model_name, {})
        rows.append({
            "family": reg.get("family", "legacy"),
            "notebook": "city_swap_batch_eval",
            "method_group": "legacy_batch_eval",
            "model_name": model_name,
            "method": None,
            "accuracy": _safe_float(metrics.get("accuracy")),
            "macro_f1": _safe_float(metrics.get("macro_f1")),
            "worst_gap": np.nan,
            "macro_gap": np.nan,
            "overall_flip_rate": _safe_float(metrics.get("overall_flip_rate")),
            "model_dir": metrics.get("model_dir", reg.get("model_dir")),
            "summary_source": str(LEGACY_RESULTS_JSON.relative_to(PROJECT_ROOT)),
        })
    return pd.DataFrame(rows)


In [ ]:
challengers_df = load_challenger_rows()
if challengers_df.empty:
    raise FileNotFoundError(
        "No challenger results found. Expected summary CSVs in figures/challengers or training_config.json files in notebooks/models/challengers."
    )

challengers_df = challengers_df.sort_values(["worst_gap", "macro_f1", "accuracy"], ascending=[True, False, False], na_position="last").reset_index(drop=True)
challengers_df.insert(0, "rank_by_gap_then_f1", np.arange(1, len(challengers_df) + 1))

challenger_out = FIGURES_DIR / "c00_challengers_comparison.csv"
challengers_df.to_csv(challenger_out, index=False)

print(f"Saved challenger table to: {challenger_out}")
challengers_df


In [ ]:
legacy_df = load_legacy_rows()
all_models_df = pd.concat([challengers_df, legacy_df], ignore_index=True, sort=False)

sort_cols = ["macro_f1", "accuracy"]
ascending = [False, False]
if "overall_flip_rate" in all_models_df.columns:
    sort_cols.append("overall_flip_rate")
    ascending.append(True)

all_models_df = all_models_df.sort_values(sort_cols, ascending=ascending, na_position="last").reset_index(drop=True)
all_models_df.insert(0, "overall_rank", np.arange(1, len(all_models_df) + 1))

all_out = FIGURES_DIR / "c00_all_models_overview.csv"
all_models_df.to_csv(all_out, index=False)

print(f"Saved overall table to: {all_out}")
all_models_df


In [ ]:
best_challenger = challengers_df.iloc[0][["model_name", "macro_f1", "worst_gap", "macro_gap", "accuracy"]]
print("Best challenger:")
print(best_challenger.to_string())

if not legacy_df.empty:
    best_legacy = legacy_df.sort_values(["macro_f1", "accuracy", "overall_flip_rate"], ascending=[False, False, True], na_position="last").iloc[0]
    print("\nBest legacy model:")
    print(best_legacy[["model_name", "macro_f1", "accuracy", "overall_flip_rate"]].to_string())
else:
    print("\nLegacy leaderboard JSON not found yet; only challenger comparison is available.")


In [ ]:
if plt is None:
    print("matplotlib is not installed; skipping plots")
else:
    plot_df = challengers_df.dropna(subset=["worst_gap", "macro_f1"]).copy()
    if plot_df.empty:
        print("No challenger rows with both gap and macro-F1 available yet.")
    else:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.scatter(plot_df["worst_gap"], plot_df["macro_f1"], s=60)
        for _, row in plot_df.iterrows():
            ax.annotate(row["model_name"], (row["worst_gap"], row["macro_f1"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
        ax.set_xlabel("Worst TPR gap (lower is better)")
        ax.set_ylabel("Macro-F1 (higher is better)")
        ax.set_title("Challengers: robustness/fairness vs quality")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plot_path = FIGURES_DIR / "c00_challengers_scatter.png"
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved plot to: {plot_path}")
